# 02 · Data cleaning: food shelf activity (`org_FSStats`)

**Goal:** one clean, trustworthy table of food shelf activity that every question (Q1, Q4, Q5, Q6, Q7) builds from.

**Input:** `Data/org_FSStats.xlsx` (one row per food shelf site per month, Jan 2022 – Jul 2026), plus `Data/Organization.xlsx` (to fill missing counties) and `Data/processed/mmg_mn_county_2022_2024.csv` (official county list + FIPS).

**Outputs (`Data/processed/`):**
1. `foodshelf_site_month.csv`: one row per site per month, cleaned, with FIPS and flags
2. `foodshelf_county_month.csv`: all 87 counties × every month, meal programs excluded
3. `foodshelf_issues_log.csv`: one row per data problem found (removed, changed, or flagged), to raise with the organizers

**Metric names used from here on:**

| Our name | Source column | Meaning |
|---|---|---|
| `visits` | `HouseholdsReg` | household trips (brief: *"one visit is one trip by one household"*) |
| `pounds` | `PoundsReg` | pounds of food distributed |
| `individuals` | `IndividualsReg` | people served (= adults + children + seniors) |
| `adults`, `children`, `seniors` | `AdultsReg`, `ChildrenReg`, `SeniorsReg` | breakdown of individuals |

📖 In-depth explanation of every step: **`docs/02_clean_foodshelf.md`**

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "Data"
OUT_DIR = DATA / "processed"

raw = pd.read_excel(DATA / "org_FSStats.xlsx")
org = pd.read_excel(DATA / "Organization.xlsx")
mmg = pd.read_csv(OUT_DIR / "mmg_mn_county_2022_2024.csv", dtype={"fips": str})
print("raw rows:", len(raw), "| columns:", raw.shape[1])

issues = []  # every removed / changed / flagged row gets logged here

def log(rows: pd.DataFrame, issue: str, action: str) -> None:
    # record problem rows (with their original numbers) in the issues log
    cols = [c for c in ["site_id", "year", "month_num", "county_original", "visits", "individuals", "pounds"] if c in rows]
    issues.append(rows[cols].assign(issue=issue, action=action))

raw rows: 28287 | columns: 58


## Step 1: Choose columns (why `…Reg`)

- **What:** keep IDs, date, county, site type, and the six `…Reg` metrics. Drop the ~40 others.
- **Why:** the data dictionary marks Extra / Produce / Holiday / Total columns **Deprecated**. The check below proves `…Total` equals `…Reg` row by row and Extra/Produce/Holiday are empty, so `…Reg` holds everything.
- **Outcome:** 58 → 13 columns, renamed to `visits`, `pounds`, `individuals`, etc.
- **Why not `…Total`:** identical numbers, but the dictionary flags it deprecated. Using the documented field is easier to defend.

In [2]:
pairs = ["Households", "Individuals", "Pounds", "Adults", "Children", "Seniors"]
check = pd.DataFrame({
    "rows where Reg != Total": [
        int((~((raw[f"{p}Reg"] == raw[f"{p}Total"]) | (raw[f"{p}Reg"].isna() & raw[f"{p}Total"].isna()))).sum())
        for p in pairs],
    "non-empty Extra+Produce+Holiday cells": [
        int(sum(raw[c].notna().sum() for c in raw.columns if c.startswith(p) and c.endswith(("Extra", "Produce", "Holiday"))))
        for p in pairs],
}, index=pairs)
check

,rows where Reg != Total,non-empty Extra+Produce+Holiday cells
Households,0,0
Individuals,0,0
Pounds,0,0
Adults,0,0
Children,1,0
Seniors,1,0


In [3]:
COLS = {
    "__ID": "row_id", "_id_site": "site_id", "_Year": "year", "_Month": "month_num",
    "County": "county_original", "AgencyType": "agency_type", "FoodBank": "food_bank",
    "HouseholdsReg": "visits", "PoundsReg": "pounds", "IndividualsReg": "individuals",
    "AdultsReg": "adults", "ChildrenReg": "children", "SeniorsReg": "seniors",
}
METRICS = ["visits", "pounds", "individuals", "adults", "children", "seniors"]
fs = raw[list(COLS)].rename(columns=COLS)
fs.head(3)

,row_id,site_id,year,month_num,county_original,agency_type,food_bank,visits,pounds,individuals,adults,children,seniors
0,2753,1254.0,2022,1,St. Louis,Brick and Mortar Site,Second Harvest Northland,1027.0,41548.0,2706.0,1519.0,1048.0,139.0
1,2762,14434.0,2022,1,Olmsted,Brick and Mortar Site,Channel One Regional Food Bank,17.0,900.0,47.0,23.0,16.0,8.0
2,12862,1069.0,2022,3,Hennepin-Minneapolis,Food Shelf,The Food Group,220.0,21345.0,343.0,281.0,27.0,35.0


## Step 2: Remove rows with an impossible month

- **What:** drop the 26 rows whose month is **0** (25 rows, northeast MN, entered in one batch on 2024-01-16) or **15** (1 row).
- **Why:** month 0 and 15 don't exist. Their `DateForSearch` points to Dec 2022 / Mar 2024, but **every one of those sites already has a real row for that month with different numbers**, so we can't tell if these are corrections, extra visits, or errors.
- **Outcome:** 26 rows removed (listed below and in the issues log). Every remaining row has a real month.
- **Why not add them to Dec 2022:** if they're re-entries, St. Louis County's Dec 2022 would be double-counted. **Why not replace:** no evidence they're more correct than the originals.

In [4]:
bad_month = ~fs["month_num"].between(1, 12)
print(f"rows with invalid month: {bad_month.sum()} | visits in them: {fs.loc[bad_month, 'visits'].sum():,.0f} "
      f"({fs.loc[bad_month, 'visits'].sum() / fs['visits'].sum():.2%} of all visits)")
display(raw.loc[bad_month.values, ["__ID", "_id_site", "_Year", "_Month", "DateForSearch", "County", "HouseholdsReg", "zzCreationTS"]].head(5))
log(fs[bad_month], "invalid month number (0 or 15)", "removed")
fs = fs[~bad_month].copy()
fs["month"] = pd.to_datetime(fs["year"].astype(str) + "-" + fs["month_num"].astype(str).str.zfill(2) + "-01")
print("date range:", fs["month"].min().date(), "->", fs["month"].max().date(), "|", fs["month"].nunique(), "months")

rows with invalid month: 26 | visits in them: 6,381 (0.05% of all visits)


,__ID,_id_site,_Year,_Month,DateForSearch,County,HouseholdsReg,zzCreationTS
10251,23361,1148.0,2023,15,2024-03-01,Polk,33.0,2023-09-18T11:26:15Z
12369,25479,14810.0,2023,0,2022-12-01,St. Louis,54.0,2024-01-16T16:03:55Z
12370,25480,1072.0,2023,0,2022-12-01,St. Louis,183.0,2024-01-16T16:03:56Z
12371,25481,14717.0,2023,0,2022-12-01,Carlton,332.0,2024-01-16T16:03:56Z
12372,25482,1103.0,2023,0,2022-12-01,St. Louis,274.0,2024-01-16T16:03:56Z


date range: 2022-01-01 -> 2026-07-01 | 55 months


## Step 3: Tag site types (`site_group`)

- **What:** group the 12 `AgencyType` labels into `food_shelf`, `outreach`, `meal`, `other`, `unknown`.
- **Why:** meal programs serve cooked meals, a different service, and they record almost nothing in these columns. Outreach delivers groceries, so it *is* food shelf-style activity.
- **Outcome:** every row keeps its data plus a `site_group`. The county-month file and Q1 **exclude only `meal`**.
- **Why not delete meal rows:** tagging is reversible. Deleting isn't, and any question can still choose.

In [5]:
SITE_GROUP = {
    "Brick and Mortar Site": "food_shelf", "Brick and Mortar Sites": "food_shelf", "Food Shelf": "food_shelf",
    "Mobile": "food_shelf", "Tribal Program": "food_shelf", "College Campus Program": "food_shelf",
    "Outreach Program": "outreach",
    "Meal Program": "meal", "Meal Programs": "meal", "On-Site Meal Programs": "meal",
    "Other Non-Profit": "other", "Agency or Tribal Nation": "other",
}
fs["site_group"] = fs["agency_type"].map(SITE_GROUP).fillna("unknown")
assert fs.loc[fs["agency_type"].notna(), "site_group"].ne("unknown").all(), "an AgencyType label is not mapped"
fs.groupby("site_group").agg(rows=("row_id", "size"), sites=("site_id", "nunique"), visits=("visits", "sum")).assign(
    visit_share_pct=lambda d: (d["visits"] / d["visits"].sum() * 100).round(1))

,rows,sites,visits,visit_share_pct
site_group,,,,
food_shelf,22885,505,11458226.0,91.3
meal,2722,66,3778.0,0.0
other,39,17,8194.0,0.1
outreach,2477,88,943429.0,7.5
unknown,138,43,130358.0,1.0


## Step 4: Fix counties → 87 official counties + FIPS

- **What:** (a) fill blank counties from `Organization.xlsx` by site ID; (b) merge `Hennepin-Minneapolis` / `Hennepin-Suburbs` → Hennepin and `Ramsey-Saint Paul` / `Ramsey-Suburbs` → Ramsey, fix `Saint Louis`, `Lac Qui Parle`; (c) attach the FIPS code; (d) drop rows still without a county.
- **Why:** Map the Meal Gap, Census and SNAP only know the 87 official counties. Without this, Hennepin and Ramsey (the two biggest counties) wouldn't join to anything.
- **Outcome:** every remaining row has `county` + `fips`. `county_original` keeps the Minneapolis/Suburbs detail.
- **Why not keep the splits:** no public dataset splits counties that way.

In [6]:
def standardize(name: pd.Series) -> pd.Series:
    s = name.str.strip().str.replace(r"-(Minneapolis|Suburbs|Saint Paul)$", "", regex=True)
    return s.replace({"Saint Louis": "St. Louis", "Lac Qui Parle": "Lac qui Parle"})

blank_before = fs["county_original"].isna().sum()
org_county = org.set_index("__ID")["County"]
fs["county_source"] = np.where(fs["county_original"].notna(), "org_FSStats", None)
fill = fs["county_original"].isna() & fs["site_id"].map(org_county).notna()
fs.loc[fill, "county_original"] = fs.loc[fill, "site_id"].map(org_county)
fs.loc[fill, "county_source"] = "Organization"
log(fs[fill], "county blank in org_FSStats", "filled from Organization.xlsx")

fs["county"] = standardize(fs["county_original"])
fips = dict(zip(mmg["county"], mmg["fips"]))
fs["fips"] = fs["county"].map(fips)

unmatched = fs.loc[fs["county"].notna() & fs["fips"].isna(), "county"].unique()
assert len(unmatched) == 0, f"county names with no FIPS match: {unmatched}"

dropped = fs[fs["fips"].isna()]
print(f"blank county before: {blank_before} | filled from Organization: {fill.sum()} | still blank -> dropped: {len(dropped)}")
print(f"dropped rows hold {dropped['visits'].sum():,.0f} visits = {dropped['visits'].sum() / fs['visits'].sum():.2%} of all visits")
log(dropped, "county blank, not in Organization.xlsx either", "removed")
fs = fs[fs["fips"].notna()].copy()
print("counties present:", fs["fips"].nunique(), "of 87 | missing:", sorted(set(mmg["county"]) - set(fs["county"])))
fs.groupby("county_original")["county"].first().loc[lambda s: s.index != s.values]

blank county before: 132 | filled from Organization: 16 | still blank -> dropped: 116
dropped rows hold 84,209 visits = 0.67% of all visits
counties present: 86 of 87 | missing: ['Wilkin']


county_original
Hennepin-Minneapolis         Hennepin
Hennepin-Suburbs             Hennepin
Lac Qui Parle           Lac qui Parle
Ramsey-Saint Paul              Ramsey
Ramsey-Suburbs                 Ramsey
Saint Louis                 St. Louis
Name: county, dtype: str

## Step 5: Fix duplicate site-months

- **What:** where a site has 2+ rows for the same month: exact copies → keep one; blank + filled → keep filled; different real numbers → **add together** and set `flag_dup_summed`.
- **Why:** one row per site per month, otherwise county totals double-count.
- **Outcome:** see the counts below. Every site-month appears once.
- **Why not "keep latest" or "keep larger":** duplicate rows share the same timestamp, and pairs like site 1103 (211 + 58 households, two months running) look like two distribution points, so dropping one loses real visits.

In [7]:
key = ["site_id", "month"]
before = len(fs)
fs["reported"] = fs[["visits", "pounds", "individuals"]].notna().any(axis=1)

# blank + filled: drop the blank row when the same site-month has a reported row
has_reported = fs.groupby(key)["reported"].transform("any")
blank_dups = fs.duplicated(key, keep=False) & ~fs["reported"] & has_reported
log(fs[blank_dups], "duplicate site-month: blank copy of a reported row", "removed blank copy")
fs = fs[~blank_dups]
# exact copies (same numbers): keep one
exact_dups = fs.duplicated(key + METRICS, keep="first")
log(fs[exact_dups], "duplicate site-month: exact copy", "removed copy")
fs = fs[~exact_dups]
# different numbers: sum them
multi = fs.duplicated(key, keep=False)
print(f"blank+filled removed: {blank_dups.sum()} | exact copies removed: {exact_dups.sum()} | "
      f"site-months with different values summed: {fs[multi].groupby(key).ngroups} ({multi.sum()} rows)")
log(fs[multi], "duplicate site-month: two different values", "summed into one row")
display(fs[multi].sort_values(key)[key + ["county", "visits", "individuals", "pounds"]].head(12))

first_cols = [c for c in fs.columns if c not in METRICS + key]
summed = fs[multi].groupby(key, as_index=False).agg(
    {**{m: lambda s: s.sum(min_count=1) for m in METRICS}, **{c: "first" for c in first_cols}})
summed["flag_dup_summed"] = True
fs = pd.concat([fs[~multi].assign(flag_dup_summed=False), summed], ignore_index=True)
assert not fs.duplicated(key).any()
print(f"rows: {before} -> {len(fs)}")

blank+filled removed: 5 | exact copies removed: 11 | site-months with different values summed: 28 (56 rows)


,site_id,month,county,visits,individuals,pounds
18412,1103.0,2024-01-01,St. Louis,211.0,541.0,11466.0
18413,1103.0,2024-01-01,St. Louis,58.0,123.0,2050.0
18438,1103.0,2024-02-01,St. Louis,212.0,782.0,11208.0
18439,1103.0,2024-02-01,St. Louis,59.0,124.0,1781.0
18464,1103.0,2024-03-01,St. Louis,215.0,563.0,10600.0
18465,1103.0,2024-03-01,St. Louis,58.0,120.0,2516.0
18490,1103.0,2024-04-01,St. Louis,215.0,558.0,10382.0
18491,1103.0,2024-04-01,St. Louis,60.0,124.0,2209.0
18516,1103.0,2024-05-01,St. Louis,200.0,543.0,10170.0
18517,1103.0,2024-05-01,St. Louis,60.0,122.0,2019.0


rows: 28145 -> 28101


## Step 6: Impossible values, typos and outliers

- **What:**
  - negative pounds/seniors → missing
  - **clear typo** = visits **> 10× that site's median month** *and* **> individuals** (and > 1,000) → visits set to missing, `flag_typo_fixed`
  - any visits/individuals/pounds **> 10× the site's median** (above a floor) that isn't a clear typo → `flag_outlier`, kept
  - any other row with visits > individuals → `flag_hh_gt_indiv`, kept
- **Why:** e.g. Waseca site 1197, Jul 2023: **273,385** households but 646 people (normal month ≈ 434). Left in, Waseca looks like the most over-served county. Ramsey site 1304, Mar 2025: households = pounds = 83,909 (pounds typed in the wrong box).
- **Outcome:** counts below. All rows are in the issues log.
- **Why not delete all >10× values:** some are real sustained spikes (Scott County site 15309, three months in a row with matching individuals and pounds). **Why not only flag:** the typos would stay in every total unless every question remembers to filter them.

In [8]:
# negatives
for m in METRICS:
    neg = fs[m] < 0
    if neg.any():
        log(fs[neg], f"negative {m}", f"{m} set to missing")
        fs.loc[neg, m] = np.nan

# typos and outliers, judged against each site's own median reported month
FLOOR = {"visits": 1000, "individuals": 1000, "pounds": 50000}
med = {m: fs.groupby("site_id")[m].transform("median") for m in FLOOR}
typo = (fs["visits"] > 10 * med["visits"]) & (fs["visits"] > fs["individuals"]) & (fs["visits"] > FLOOR["visits"])
display(fs.loc[typo, ["site_id", "county", "month", "visits", "individuals", "pounds"]].assign(site_median_visits=med["visits"][typo]))
log(fs[typo], "visits typo: >10x site median and more households than individuals", "visits set to missing")
fs["flag_typo_fixed"] = typo
fs.loc[typo, "visits"] = np.nan

outlier = pd.Series(False, index=fs.index)
for m, floor in FLOOR.items():
    outlier |= (fs[m] > 10 * med[m]) & (fs[m] > floor)
fs["flag_outlier"] = outlier
log(fs[outlier], "value >10x site median (not a clear typo)", "flagged, kept")

fs["flag_hh_gt_indiv"] = fs["visits"] > fs["individuals"]
log(fs[fs["flag_hh_gt_indiv"]], "more households than individuals", "flagged, kept")

print(f"typos fixed: {typo.sum()} | outliers flagged: {outlier.sum()} | households > individuals flagged: {fs['flag_hh_gt_indiv'].sum()} "
      f"({fs.loc[fs['flag_hh_gt_indiv'], 'visits'].sum() / fs['visits'].sum():.2%} of visits)")

,site_id,county,month,visits,individuals,pounds,site_median_visits
9075,1245.0,Polk,2023-06-01,2247.0,706.0,36550.0,220.0
9539,1197.0,Waseca,2023-07-01,273385.0,646.0,12053.0,434.0
14045,15306.0,Dakota,2024-04-01,3257.0,371.0,3257.0,63.0
17179,1083.0,Hennepin,2024-11-01,4203.0,1882.0,27039.0,400.0
19795,1304.0,Ramsey,2025-03-01,83909.0,13735.0,83909.0,2708.0
21534,1277.0,Marshall,2025-07-01,1590.0,49.0,NaN,22.0
23488,1417.0,Hubbard,2025-11-01,6670.0,239.0,NaN,78.0


typos fixed: 7 | outliers flagged: 43 | households > individuals flagged: 145 (0.82% of visits)


## Step 7: Blank vs. zero months

- **What:** keep blanks as **missing** (`reported = False`), keep zeros as zero.
- **Why:** blank = "didn't send numbers", zero = "reported nobody came". Treating blank as 0 makes counties look falsely under-served, which is the exact thing Q1 looks for.
- **Outcome:** each row has `reported`. The county file counts missing site-months so gaps are visible.
- **Why not fill blanks with a guess:** that invents data. We may impute later for Q7, explicitly.

In [9]:
fs["reported"] = fs[["visits", "pounds", "individuals"]].notna().any(axis=1)
print(f"site-months reported: {fs['reported'].sum():,} | not reported (blank): {(~fs['reported']).sum():,} | "
      f"reported zero visits: {(fs['visits'] == 0).sum():,}")
fs.groupby(fs["month"].dt.year)["reported"].mean().mul(100).round(1).rename("% site-months reported")

site-months reported: 26,173 | not reported (blank): 1,928 | reported zero visits: 1,577


month
2022    92.3
2023    93.2
2024    93.9
2025    92.7
2026    94.0
Name: % site-months reported, dtype: float64

## Step 8: Save the site-month file and the issues log

- `foodshelf_site_month.csv`: one row per site per month, all site types, with `flag_…` columns. **The master file** everything else builds from.
- `foodshelf_issues_log.csv`: every row we removed, changed or flagged, with the original numbers, the problem and what we did. Use it to raise issues with the organizers.

In [10]:
SITE_COLS = ["site_id", "month", "year", "month_num", "fips", "county", "county_original", "county_source",
             "agency_type", "site_group", "food_bank", *METRICS, "reported", "flag_dup_summed", "flag_typo_fixed", "flag_outlier", "flag_hh_gt_indiv"]
site_month = fs[SITE_COLS].sort_values(["fips", "site_id", "month"]).reset_index(drop=True)
site_month.to_csv(OUT_DIR / "foodshelf_site_month.csv", index=False)
print("saved foodshelf_site_month.csv:", site_month.shape)

log_df = pd.concat(issues, ignore_index=True)
log_df.to_csv(OUT_DIR / "foodshelf_issues_log.csv", index=False)
print("saved foodshelf_issues_log.csv:", len(log_df), "rows")
log_df.groupby(["issue", "action"]).size().rename("rows").to_frame()

saved foodshelf_site_month.csv: (28101, 22)
saved foodshelf_issues_log.csv: 430 rows


,,rows
issue,action,
county blank in org_FSStats,filled from Organization.xlsx,16
"county blank, not in Organization.xlsx either",removed,116
duplicate site-month: blank copy of a reported row,removed blank copy,5
duplicate site-month: exact copy,removed copy,11
duplicate site-month: two different values,summed into one row,56
invalid month number (0 or 15),removed,26
more households than individuals,"flagged, kept",145
negative pounds,pounds set to missing,4
negative seniors,seniors set to missing,1


## Step 9: County-month file (meal programs excluded)

- **What:** add up sites to **county × month**, for **all 87 counties × every month** (a county with no sites gets `n_sites = 0`, e.g. Wilkin).
- **Why:** Q1 (by year), Q4/Q5 (monthly patterns) and Q7 (August forecast) all work at county level.
- **Outcome:** metrics summed over *reported* sites, plus `n_sites`, `n_sites_reported`, `n_sites_missing`. A county-month where nothing was reported gets blank metrics, not 0.
- **Why not include meal programs:** they're a different service with ~0 recorded activity (Step 3).

In [11]:
use = site_month[site_month["site_group"] != "meal"]
agg = use.groupby(["fips", "month"]).agg(
    **{m: (m, lambda s: s.sum(min_count=1)) for m in METRICS},
    n_sites=("site_id", "nunique"),
    n_sites_reported=("reported", "sum"),
)
grid = pd.MultiIndex.from_product([sorted(mmg["fips"].unique()), sorted(site_month["month"].unique())], names=["fips", "month"])
county_month = agg.reindex(grid).reset_index()
county_month[["n_sites", "n_sites_reported"]] = county_month[["n_sites", "n_sites_reported"]].fillna(0).astype(int)
county_month["n_sites_missing"] = county_month["n_sites"] - county_month["n_sites_reported"]
county_month["county"] = county_month["fips"].map(dict(zip(mmg["fips"], mmg["county"])))
county_month = county_month[["fips", "county", "month", *METRICS, "n_sites", "n_sites_reported", "n_sites_missing"]]
county_month.to_csv(OUT_DIR / "foodshelf_county_month.csv", index=False)
print("saved foodshelf_county_month.csv:", county_month.shape, "=", county_month["fips"].nunique(), "counties x", county_month["month"].nunique(), "months")
print("counties with no sites at all:", sorted(county_month.groupby("county")["n_sites"].sum().loc[lambda s: s == 0].index))

saved foodshelf_county_month.csv: (4785, 12) = 87 counties x 55 months
counties with no sites at all: ['Wilkin']


## Step 10: Sanity checks

Statewide totals by year, and a comparison with The Food Group's published visit counts (`TFG_SlideDeck.pdf`, slide 7). Their "visits" match **individuals**, not households (see the Visits definition decision in `docs/02_clean_foodshelf.md`).

In [12]:
yearly = county_month.groupby(county_month["month"].dt.year)[["visits", "individuals", "pounds"]].sum().astype(int)
yearly["months"] = county_month.groupby(county_month["month"].dt.year)["month"].nunique()
display(yearly)

published = pd.Series({"2025-01": 775_832, "2025-02": 719_103, "2025-03": 756_863, "2025-04": 782_437,
                       "2026-01": 717_575, "2026-02": 850_781, "2026-03": 859_076, "2026-04": 823_884}, name="TFG published visits")
ours = site_month.groupby(site_month["month"].dt.strftime("%Y-%m"))[["visits", "individuals"]].sum()
cmp = pd.concat([published, ours], axis=1, join="inner")
cmp["individuals / published"] = (cmp["individuals"] / cmp["TFG published visits"]).round(3)
cmp

,visits,individuals,pounds,months
month,,,,
2022,1963236,5626130,118672911,12
2023,2482146,7625430,139985146,12
2024,2862391,8994350,155558493,12
2025,3016327,9055404,151105766,12
2026,1755606,5422702,89162850,7


,TFG published visits,visits,individuals,individuals / published
2025-01,775832,247974.0,778384.0,1.003
2025-02,719103,241336.0,722068.0,1.004
2025-03,756863,244459.0,760491.0,1.005
2025-04,782437,263231.0,786971.0,1.006
2026-01,717575,238004.0,716558.0,0.999
2026-02,850781,268283.0,853046.0,1.003
2026-03,859076,279131.0,865523.0,1.008
2026-04,823884,263173.0,801510.0,0.973
